# Chapter 9: Self-Supervised Learning

This notebook accompanies **Chapter 9** of the lecture notes.

> Last lecture the encoder arrived pretrained, someone else paid for it, and the question was how to reuse it. Today there is no encoder. We have a *corpus* — a body of raw data that nobody has labelled — and the only supervision available is whatever signal we can extract from the data itself. Mask part of the input and predict it; mask a span and predict the span; mask the future and predict the next token. The same principle, three different masking patterns. The representation comes for free.

**Agenda**

🏷️ · 🎭 · 🧩 · 📜 · 🪞 · 🏁

**Take it from here:** 🧵

> **Tip:** Run cells top to bottom. Later cells depend on earlier ones. Each section trains a tiny transformer (~3000 parameters) on a synthetic banana-sentence corpus; everything fits inside this kernel and runs in well under a minute per section.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, '../..')
from plot_style import *
from checks import (
    check_attention, check_mask_tokens, check_masked_token_loss,
    check_geometric_span_mask, check_causal_mask, check_next_token_loss,
    check_temperature_sample, check_ema_update, check_jepa_loss,
)
from viz_helpers import (
    load_banana_corpus, build_word_vocab, encode_words, decode_ids, visible_vocab,
    init_params, forward, train_lm, train_jepa_run, sample,
    plot_loss_curve, plot_attention_internals, plot_attention_masks,
    show_fill_in_examples,
    show_span_vs_single, plot_jepa_variance,
    show_one_sentence_many_pairs, show_three_pretext_tasks,
    plot_label_gap_magnitude,
    DEFAULT_D_MODEL, DEFAULT_CONTEXT, MASK_ID,
)

RNG = np.random.default_rng(0)

## 🏷️ The Label Gap

A *dataset* is curated and usually labelled, assembled with a specific task in mind. A *corpus* is the body of raw data itself, often scraped or harvested at web scale, with no task attached. Self-supervised learning is what lets a corpus take the place of a dataset — the supervision is read off the structure of the data instead of being written down by an annotator.

> ImageNet absorbed years of human effort to assemble fourteen million labelled images. The open web holds a hundred billion images that no curator will ever touch. If a model only learns from labelled examples, what is its scale ceiling?

<details><summary>Thought</summary>

The annotation budget. Labels are expensive, biased by the curator, slow to scale with model size, and tied to the specific domain that worked them out. A model whose only signal is hand-written labels grows exactly as fast as people can produce them. The asymmetry between labelled datasets and raw corpora is the structural fact that motivates this chapter: scale is bounded by storage, not by human effort, only if we can extract a training signal from the corpus itself.
</details>


In [ ]:
plot_label_gap_magnitude()

### One sentence, many training pairs

Take a single sentence from the corpus we are about to use. By choosing where to mask, we generate as many *(input, target)* pairs as the sentence has words — and never write down a label.


In [ ]:
show_one_sentence_many_pairs(
    'she slipped on a yellow banana peel and fell over',
    n_pairs=9,
)


### Three pretext tasks, same sentence

The same sentence is the supervision for three different self-supervised objectives. The rest of this notebook is each of these in turn.


In [ ]:
show_three_pretext_tasks('she slipped on a yellow banana peel and fell over')


**Observe:**

- One sentence yields *N* training pairs by masking each position in turn — the corpus *is* the label set.
- Single-token, span, and next-token shifts are three ways to read supervision off the same string. They lead to encoder-only, encoder-decoder, and decoder-only models respectively.
- The handcrafted vision pretext tasks (rotate the image and predict the angle, scramble the patches and reassemble, fill in colour) follow the same pattern: pick a transformation the data already supports, and the inverse is the label.


## 🎭 Masked-Token Denoising (BERT)

The simplest pretext task on text is the cloze: cover up a token, ask the model to fill it in. The network has to develop an internal sense of which tokens are likely in any given context, and the only label needed is the original token, which the data itself provides. This is BERT's recipe at its core.

A masked language model uses **unrestricted attention**: when predicting a masked position, the model is allowed to see tokens both before and after it. Every position can look at every other position, including itself — there is no mask on the attention pattern. The causal mask of the next section is the special case that hides the future; plain attention has no such restriction.

We train on a synthetic corpus of **banana sentences** — short repetitive English sentences such as *"she slipped on a yellow banana peel"*. Word-level tokenisation, vocabulary of about thirty words, just over three hundred token positions. Small enough that a tiny transformer learns the patterns in seconds; structured enough that the demos visibly succeed.

In [ ]:
text = load_banana_corpus()
stoi, itos = build_word_vocab(text)
vocab_size = len(stoi)
ids = encode_words(text, stoi)

print(f'Corpus    : {len(ids)} word tokens')
print(f'Vocab     : {vocab_size} words  (slot 0 = [MASK])')
print()
print('First 8 sentences:')
for line in text.split(' the ', 8)[:8]:
    print('  the ' + line if not line.startswith('she') else '  ' + line)
print()
print('First 20 vocab entries:')
print(visible_vocab(stoi, n=20))


### The attention mechanism

The attention mechanism is the centrepiece of every transformer in this notebook. Three tensors `Q` (queries), `K` (keys), and `V` (values), all with shape `(B, T, D)` — a batch of `B` sequences, each `T` positions long, with `D`-dimensional embeddings — are derived from the input. The output at position `i` is a weighted sum of the value vectors, where the weights come from a softmax over the dot products between query `i` and every key.

> Why is the dot product divided by `sqrt(D)` before softmax? What goes wrong if you skip that step?

<details><summary>Thought</summary>

For random unit vectors in `D` dimensions the dot product has variance roughly `D`. Without dividing by `sqrt(D)` the scores grow with the embedding dimension, the softmax saturates (one token gets all the weight, every other gets zero), and the gradient through that softmax collapses. Dividing by `sqrt(D)` keeps the scores in a regime where the softmax has gradient on more than one entry; it is the only modification of "raw" dot-product attention in scaled dot-product attention.
</details>

Implement `attention(Q, K, V, attn_mask=None)`. It should compute scaled dot-product attention. When `attn_mask` is `None`, every position attends to every other. When `attn_mask` is provided, it is an additive mask of shape `(T, T)` — `0` for allowed cells and `-inf` for blocked cells — added to the scores before the softmax.

Useful: `np.swapaxes(K, -1, -2)` to transpose the last two axes; `np.exp`, `.sum(axis=-1, keepdims=True)` for softmax; subtract the per-row max before exponentiating for numerical stability.

In [ ]:
def attention(Q, K, V, attn_mask=None):
    """Scaled dot-product attention.

    Parameters
    ----------
    Q, K, V    : ndarray of shape (B, T, D)
    attn_mask  : ndarray of shape (T, T), additive (0 allowed, -inf blocked), or None.

    Returns
    -------
    ndarray of shape (B, T, D)
    """
    # 1. scores = Q @ K.T / sqrt(D)            (broadcast over batch)
    # 2. if attn_mask is not None:  scores = scores + attn_mask
    # 3. weights = softmax(scores, axis=-1)    (subtract max for stability)
    # 4. return weights @ V
    # YOUR CODE HERE
    pass


check_attention(attention)


### Inside attention: scores and weights

The two `T × T` matrices that come out of the block above, on the first eight words of a sample sentence. Q and K (each `T × D`) are random projections, so their values are arbitrary - but `Q · Kᵀ / √D` collapses them into a meaningful score matrix, and the row-wise softmax sharpens it into the weights that mix V. This is the **flow** your `attention(Q, K, V)` computes.

In [ ]:
plot_attention_internals()

### Mask 30% of the tokens — BERT's 80/10/10 recipe

The original BERT paper masks 15% of input tokens. Of those *chosen* positions, 80% are replaced with the `[MASK]` token, 10% are replaced with a *random* token from the vocabulary, and 10% are *kept unchanged*. The mixture matters: training only with `[MASK]` would create a distribution gap between training (where the symbol appears) and inference (where it does not).

> The chapter cites 15%. We will train at 30%. Why the higher rate here?

<details><summary>Thought</summary>

Two effects compound at our scale. First, our corpus has only ~350 word tokens, so 15% gives a thin supervision signal per batch — context routing through attention barely starts to differentiate positions. Second, the 80/10/10 mixture means only ~80% of *chosen* positions are actually replaced with `[MASK]`, so at 15% only ~12% of positions ever see the special symbol during training. With a higher mask fraction, more positions see `[MASK]` during training, the model gets more contextual supervision per batch, and the distribution it sees at inference time (one `[MASK]` per query) is closer to the training distribution.
</details>

Implement `mask_tokens(ids, mask_id, vocab_size, frac, rng)`. Return a tuple `(corrupted_ids, mask_positions)` where `mask_positions` is a boolean array of shape `ids.shape` marking the positions chosen for prediction (True at the chosen fraction, False elsewhere).

Useful: `rng.random(shape) < frac` for boolean selection; `rng.integers(low, high, size)` for random tokens; `np.where` to assemble the corrupted output.


In [ ]:
def mask_tokens(ids, mask_id, vocab_size, frac, rng):
    """Apply BERT-style 80/10/10 masking. ids has shape (B, T)."""
    # 1. chosen = rng.random(ids.shape) < frac           # bool (B, T)
    # 2. sub    = rng.random(ids.shape)                  # for 80/10/10 split
    # 3. is_mask  = chosen & (sub < 0.8)
    #    is_rand  = chosen & (sub >= 0.8) & (sub < 0.9)
    # 4. corrupted = ids.copy()
    #    corrupted[is_mask] = mask_id
    #    rand_ids = rng.integers(1, vocab_size, ids.shape)   # avoid mask_id (0)
    #    corrupted[is_rand] = rand_ids[is_rand]
    # 5. return corrupted, chosen
    # YOUR CODE HERE
    pass


check_mask_tokens(mask_tokens)


### Cross-entropy at the masked positions only

The loss is cross-entropy, but only at the positions we chose to mask. Other positions are not predicted (they were not corrupted), so they should not contribute to the gradient.

> The model produces logits for *every* position. Why is it important to compute the loss at the masked positions only, and not the cross-entropy averaged over all `B*T` positions?

<details><summary>Thought</summary>

Two reasons. The first is mechanical: only at masked positions is the model actually being asked to predict something it does not already see. At non-masked positions the input *contains* the answer, so the model can copy it through the residual stream — the loss there is zero by construction once the model has learned the trivial copy, and contributes no useful gradient. The second is interpretive: averaging over all positions dilutes the signal at masked positions by a factor of `1/frac`, which both slows learning and confuses metrics — the per-token loss of an unmasked position is meaningless.
</details>

Implement `masked_token_loss(logits, targets, mask_positions)`. The shapes are `(B, T, V)` for logits and `(B, T)` for targets — `V` is the vocabulary size, the axis along which the model's predicted distribution lies. Use a numerically stable log-softmax (subtract the max across the vocab axis before exponentiating).


In [ ]:
def masked_token_loss(logits, targets, mask_positions):
    """Cross-entropy at positions where mask_positions is True.

    Parameters
    ----------
    logits         : (B, T, V)
    targets        : (B, T) int
    mask_positions : (B, T) bool
    """
    # 1. m = logits.max(axis=-1, keepdims=True)
    # 2. log_probs = logits - m - log(sum(exp(logits - m), axis=-1, keepdims=True))
    # 3. flat_logp = log_probs[mask_positions]      # (n_masked, V)
    #    flat_tgt  = targets[mask_positions]        # (n_masked,)
    # 4. return -mean(flat_logp[arange(n_masked), flat_tgt])
    # YOUR CODE HERE
    pass


check_masked_token_loss(masked_token_loss)


### Train

`train_lm` runs Adam on a tiny one-block transformer. It calls into your `attention` for the forward pass; the gradient is computed analytically inside the helper. About 1500 Adam steps drives the loss from the uniform baseline (~3.5) to under 1.0 — the model has actually learned the corpus.

> What is the *uniform baseline* drawn on every loss curve, and why is it placed at `log(vocab_size)`?

<details><summary>Thought</summary>

It is the cross-entropy loss of a model that predicts every token with equal probability `1/V`. Cross-entropy is `-log(P(target))`; if the model spreads probability uniformly then `P(target) = 1/V`, so the loss is `-log(1/V) = log(V)`. For our 34-word vocabulary, `log(34) ≈ 3.53`. That is what an untrained network produces, so a training curve that sits *on* the baseline has not learned anything — not even that *banana* appears more often than *farmer*. Dropping below the baseline means the model has at least picked up unigram statistics; dropping well below it (toward zero) means the model has started to use context.
</details>


In [ ]:
params_mlm = init_params(vocab_size, seed=0)

def _mlm_mask_strategy(batch, rng):
    return mask_tokens(batch, MASK_ID, vocab_size, frac=0.30, rng=rng)

if attention(np.zeros((1, 4, DEFAULT_D_MODEL), dtype=np.float32),
             np.zeros((1, 4, DEFAULT_D_MODEL), dtype=np.float32),
             np.zeros((1, 4, DEFAULT_D_MODEL), dtype=np.float32)) is None:
    print('⬜ Implement attention above first.')
elif masked_token_loss(np.zeros((1, 1, vocab_size), dtype=np.float32),
                       np.zeros((1, 1), dtype=np.int64),
                       np.array([[True]])) is None:
    print('⬜ Implement masked_token_loss above first.')
else:
    params_mlm, losses_mlm = train_lm(
        params_mlm, ids, attention,
        mask_strategy_fn=_mlm_mask_strategy,
        use_causal_mask=False,
        n_steps=1500, batch_size=64, lr=0.01, seed=0,
    )
    plot_loss_curve(losses_mlm, 'Masked language model training',
                    baseline=float(np.log(vocab_size)))

### Fill in the blanks

Mask one token at a time in a held-out banana sentence and read off the model's top prediction. The textbook example, *"she slipped on a yellow [_] peel and fell over"* should come back as *banana*.


In [ ]:
if attention(np.zeros((1, 4, DEFAULT_D_MODEL), dtype=np.float32),
             np.zeros((1, 4, DEFAULT_D_MODEL), dtype=np.float32),
             np.zeros((1, 4, DEFAULT_D_MODEL), dtype=np.float32)) is None:
    print('⬜ Need attention implemented to run the demo.')
else:
    show_fill_in_examples(params_mlm, ids, itos, attention, _mlm_mask_strategy,
                          n_examples=4, seed=2)

**Observe:**

- After 1500 Adam steps the model fills in masked positions with real words. *yellow* `[_]` *peel* comes back as *banana*; *the monkey* `[_]` *a yellow banana* comes back as *ate* or *peeled*.
- The model sees both the left and the right context at each masked position. Try predicting from the left only and the task gets noticeably harder. The next section's causal model will do exactly that.
- The 80/10/10 mixture at 30% mask rate is BERT's recipe scaled to a tiny corpus. At a real scale (BERT-base on Wikipedia, vocabulary ~30k), the mixture matters because the model needs to behave consistently when `[MASK]` does not appear at inference.

## 🧩 Span Masking (SpanBERT, T5)

Single-token masking is often too easy: with twenty visible neighbours, the model can interpolate the missing word from the local n-gram statistics. The harder pretext task is **span masking**: hide a *contiguous chunk* of several tokens at once, so that the model has to reconstruct multi-token structure from the surrounding context.

> What is the model forced to learn from span masking that it cannot learn from single-token masking, even at the same overall mask fraction?

<details><summary>Thought</summary>

Single-token masking can be solved by short-range copying from the immediate neighbours — *yellow* `[_]` *peel* is almost always *banana* given the two visible words. Span masking removes a contiguous block — *yellow* `[_ _ _]` *and fell over* — so the model has to reconstruct the whole phrase together: it cannot copy the missing words from their direct neighbours because those neighbours are also missing. The model is forced to integrate longer-range context (the rest of the sentence) and to model the *joint* distribution of the masked tokens, not just their marginals one at a time.
</details>

Implement `geometric_span_mask(ids, mask_id, mean_span, mask_frac, rng)`. Pick span starts uniformly at random; for each start, sample a span length from a geometric distribution with mean `mean_span`; mask the contiguous run by replacing those positions with `mask_id`. Stop when roughly `mask_frac * T` positions have been masked. Return `(corrupted_ids, mask_positions)`.

Useful: `rng.geometric(p, size)` returns geometric samples with parameter `p = 1/mean`; clip the span end so it does not exceed `T`; check overlap with already-masked positions and skip on collision.


In [ ]:
def geometric_span_mask(ids, mask_id, mean_span, mask_frac, rng):
    """Span-mask each row of ids. Returns (corrupted, mask_positions)."""
    # B, T = ids.shape
    # target_n = int(mask_frac * T)
    # mask_positions = np.zeros_like(ids, dtype=bool)
    # for b in range(B):
    #     masked = 0
    #     while masked < target_n:
    #         length = max(1, int(rng.geometric(1.0 / mean_span)))
    #         length = min(length, target_n - masked)
    #         start  = int(rng.integers(0, max(1, T - length + 1)))
    #         if mask_positions[b, start:start + length].any():
    #             continue
    #         mask_positions[b, start:start + length] = True
    #         masked += length
    # corrupted = ids.copy()
    # corrupted[mask_positions] = mask_id
    # return corrupted, mask_positions
    # YOUR CODE HERE
    pass


check_geometric_span_mask(geometric_span_mask)


### Same window, two masking strategies


In [ ]:
def _single_demo(batch, rng):
    return mask_tokens(batch, MASK_ID, vocab_size, frac=0.25, rng=rng)

def _span_demo(batch, rng):
    return geometric_span_mask(batch, MASK_ID, mean_span=3.0,
                                mask_frac=0.25, rng=rng)

if (mask_tokens(ids[:DEFAULT_CONTEXT][None, :], MASK_ID, vocab_size, 0.15,
                np.random.default_rng(0)) is not None
    and geometric_span_mask(ids[:DEFAULT_CONTEXT][None, :], MASK_ID, 3.0, 0.20,
                             np.random.default_rng(0)) is not None):
    show_span_vs_single(ids, itos, _single_demo, _span_demo,
                        context=DEFAULT_CONTEXT, seed=4)
else:
    print('⬜ Need both mask_tokens and geometric_span_mask implemented.')


### Train with span masking

Same architecture, same trainer, same loss; only the masking strategy changes. The span-masked task is genuinely harder than single-token masking at the same overall mask fraction.


In [ ]:
params_sp = init_params(vocab_size, seed=1)

def _sp_mask_strategy(batch, rng):
    return geometric_span_mask(batch, MASK_ID, mean_span=3.0,
                                mask_frac=0.30, rng=rng)

if (geometric_span_mask(ids[:DEFAULT_CONTEXT][None, :], MASK_ID, 3.0, 0.30,
                         np.random.default_rng(0)) is not None
    and masked_token_loss(np.zeros((1, 1, vocab_size), dtype=np.float32),
                          np.zeros((1, 1), dtype=np.int64),
                          np.array([[True]])) is not None):
    params_sp, losses_sp = train_lm(
        params_sp, ids, attention,
        mask_strategy_fn=_sp_mask_strategy,
        use_causal_mask=False,
        n_steps=1500, batch_size=64, lr=0.01, seed=1,
    )
    plot_loss_curve(losses_sp, 'Span-masked language model training',
                    baseline=float(np.log(vocab_size)))
else:
    print('⬜ Need geometric_span_mask and masked_token_loss implemented first.')


**Observe:**

- The span-masked loss starts and ends higher than the single-token loss. The task is genuinely harder, exactly because the model cannot lean on local interpolation.
- After the same number of optimisation steps the span-masked model has a higher per-position loss, but a meaningful one — it has been forced to use longer-range context to recover whole phrases like *yellow banana peel*.
- This is the same mechanism that, at GPT-3 scale, makes span corruption the default in encoder-decoder pretraining: harder pretext, better representation.


## 📜 Causal Next-Token Prediction (GPT)

Decoder-only language models take the same denoising principle and apply it strictly left to right. At every position, the model sees only the prefix and predicts the next token. The mask is no longer "predict the holes" but "you can never look at the future". The supervision signal is everywhere: at each of `T` positions, the next token is the label.

> The masked language model trained on a small fraction of the tokens (15–30%) at each step. The causal language model trains on EVERY position. Why is the causal model not therefore strictly better?

<details><summary>Thought</summary>

Two reasons, both about what the model is asked to predict. The masked language model at any masked position can use both left and right context, which is a richer signal per prediction; the causal language model at position `i` can only use the prefix `0..i-1`. The causal model has more predictions per pass, but each one is harder. Whether that trade-off favours one or the other depends on the downstream task: if you want to *generate* text, you need a left-to-right model; if you want to *understand* text (classification, retrieval, fill-in), the masked encoder tends to be a better representation per parameter. The two recipes have therefore split into two architectural families.
</details>

### The causal mask

The causal mask is an additive `(T, T)` matrix: `0` for cells where the query is allowed to attend to the key (`j ≤ i`), and `-inf` for cells in the future (`j > i`). Adding this to the dot-product scores before the softmax sets the future weights to zero exactly.

Implement `causal_mask(T)`.

Useful: `np.zeros((T, T))`; `np.triu(..., k=1)` keeps the strict upper triangle; assign `-inf` there.


In [ ]:
def causal_mask(T):
    """Return (T, T) additive mask: 0 where j <= i, -inf where j > i."""
    # mask = np.zeros((T, T), dtype=np.float32)
    # upper = np.triu(np.ones((T, T), dtype=bool), k=1)   # strict upper triangle
    # mask[upper] = -np.inf
    # return mask
    # YOUR CODE HERE
    pass


check_causal_mask(causal_mask)


### Picture: encoder vs causal attention

In [ ]:
if causal_mask(4) is not None:
    plot_attention_masks(context=8)
else:
    print('⬜ Need causal_mask implemented to render the diagram.')


### Next-token cross-entropy

At every position `t`, the model predicts position `t+1`. The loss is the cross-entropy of the predicted distribution against the actual next token, averaged over all but the last position (which has no follower to predict).

Implement `next_token_loss(logits, targets)` where `logits` has shape `(B, T, V)` and `targets` has shape `(B, T)`. The prediction at position `t` corresponds to `targets[t+1]`, so use `logits[:, :-1, :]` against `targets[:, 1:]`.


In [ ]:
def next_token_loss(logits, targets):
    """Cross-entropy of next-token prediction over a window."""
    # 1. pred_logits = logits[:, :-1, :]                # (B, T-1, V)
    # 2. pred_targets = targets[:, 1:]                  # (B, T-1)
    # 3. log_probs = stable_log_softmax(pred_logits)
    # 4. return -mean(log_probs gathered at pred_targets)
    # YOUR CODE HERE
    pass


check_next_token_loss(next_token_loss)


### Train the causal language model

Same architecture as 🎭, with the causal mask switched on and the loss switched to `next_token_loss`. Because every position in every window contributes a supervision signal, the causal language model trains much faster than the masked one — about ten times fewer Adam steps suffice.


In [ ]:
params_cs = init_params(vocab_size, seed=2)

if (causal_mask(4) is not None
    and next_token_loss(np.zeros((1, 2, vocab_size), dtype=np.float32),
                        np.zeros((1, 2), dtype=np.int64)) is not None):
    params_cs, losses_cs = train_lm(
        params_cs, ids, attention,
        mask_strategy_fn=None,           # causal: predict every shifted position
        use_causal_mask=True,
        n_steps=200, batch_size=32, lr=0.05, seed=2,
    )
    plot_loss_curve(losses_cs, 'Causal language model training',
                    baseline=float(np.log(vocab_size)))
else:
    print('⬜ Need causal_mask and next_token_loss implemented first.')


### Sampling: temperature controls how confident the model behaves

A trained causal language model defines a probability distribution over the next token. *Temperature* sampling rescales the logits by a constant `T` before the softmax. Low temperature concentrates probability mass on the top candidates; high temperature flattens the distribution and lets less-likely tokens win.

Implement `temperature_sample(logits, temperature, rng)`. Returns a single integer token id sampled from the rescaled distribution.


In [ ]:
def temperature_sample(logits, temperature, rng):
    """Sample one token id from the categorical distribution defined by logits / temperature."""
    # 1. scaled = logits / max(temperature, 1e-6)
    # 2. probs  = softmax(scaled)
    # 3. return int(rng.choice(len(probs), p=probs))
    # YOUR CODE HERE
    pass


check_temperature_sample(temperature_sample)


### Sample some banana-flavoured continuations

Three different prefixes, the same model, three different distributions over what comes next. At low temperature the model is conservative and rolls out the trigram patterns it has memorised; at high temperature the entropy of each step rises and the output drifts.


In [ ]:
if (causal_mask(4) is not None
    and temperature_sample(np.array([0.1, 0.5, -0.2, 1.0]), 1.0,
                           np.random.default_rng(0)) is not None):
    for prefix in ('she slipped on a yellow', 'the monkey ate a',
                   'he peeled the', 'they found a yellow'):
        seed_ids = encode_words(prefix, stoi)
        out = sample(params_cs, seed_ids, attention,
                     max_new_tokens=8, temperature=0.5, seed=0)
        cont = decode_ids(out[len(seed_ids):], itos)
        print(f'  T=0.5  {prefix!r:>32} -> {cont!r}')
    print()
    for prefix in ('she slipped on a yellow', 'the monkey ate a'):
        seed_ids = encode_words(prefix, stoi)
        out = sample(params_cs, seed_ids, attention,
                     max_new_tokens=8, temperature=1.5, seed=0)
        cont = decode_ids(out[len(seed_ids):], itos)
        print(f'  T=1.5  {prefix!r:>32} -> {cont!r}')
else:
    print('⬜ Need causal_mask and temperature_sample implemented first.')


### Prompt steering

The simplest form of conditioning. Three different subjects; the same model; three different distributions of continuations. The model never *knew* that any prefix was special — it just continues whatever pattern the prefix establishes.


In [ ]:
if temperature_sample(np.array([0.1, 0.5, -0.2, 1.0]), 1.0,
                       np.random.default_rng(0)) is not None:
    for subject in ('the cat', 'the monkey', 'the baker', 'the farmer'):
        seed_ids = encode_words(subject + ' ate', stoi)
        out = sample(params_cs, seed_ids, attention,
                     max_new_tokens=6, temperature=0.5, seed=0)
        cont = decode_ids(out[len(seed_ids):], itos)
        print(f'  {subject + " ate":>22} -> {cont!r}')
else:
    print('⬜ Need temperature_sample.')


### Prompt injection

Once the model is trained left to right, a prefix is the *only* steering tool. A name in front of the generated continuation biases every following token toward whatever pattern the prefix establishes. At small scale this is *prompt steering*. At large scale, the same mechanic becomes *prompt injection* — a deliberately crafted prefix overrides the developer's intended behaviour and causes the system to follow whatever instruction the attacker put in front of it.

> The model was trained to predict the next token given a prefix. Why is "follow the instruction in the prompt" the same operation as "complete the prefix"?

<details><summary>Thought</summary>

The model has no separate channel for instructions: *everything* it sees is the prefix. If the training data contained patterns where an instruction-like phrase was followed by something that looked like compliance (which web-scale corpora do, abundantly), then continuing those patterns at inference time IS following an instruction. The model is not "deciding" whether to obey; it is sampling the most probable continuation, and instruction-following is the highest-probability continuation in many contexts. This is why prompt injection is hard to fix with prompts alone — to the model, the malicious instruction and the developer's instruction have the same shape.
</details>

We demonstrate the *mechanism* with a style switch: continue from one subject, splice a different subject mid-stream, watch the distribution flip.


In [ ]:
if temperature_sample(np.array([0.1, 0.5, -0.2, 1.0]), 1.0,
                       np.random.default_rng(0)) is not None:
    stage1 = encode_words('the cat', stoi)
    out1   = sample(params_cs, stage1, attention,
                    max_new_tokens=6, temperature=0.5, seed=0)
    print('STAGE 1  the cat ...')
    print('         ' + repr(decode_ids(out1, itos)))
    print()

    # Splice a different prefix mid-stream and resume.
    injection = encode_words('the monkey', stoi)
    stage2 = np.concatenate([out1, injection])
    out2   = sample(params_cs, stage2, attention,
                    max_new_tokens=6, temperature=0.5, seed=0)
    print('STAGE 2  ... the monkey  (after injection)')
    print('         ' + repr(decode_ids(out2[len(stage1):], itos)))
else:
    print('⬜ Need temperature_sample.')


**Observe:**

- Low temperature *should* roll out the trigram patterns the model memorised — *yellow → banana → peel*, *ate a → ripe banana* — while higher temperature *would* flatten the distribution and let less-likely tokens through. On a 30-word corpus the patterns are thin, so the difference can be subtle; at web scale this is the "creativity vs coherence" knob in full force.
- Different prefixes *should* elicit different distributions even though the model has no notion of "subject" — *the baker ate* and *the monkey ate* would in principle steer toward different completions. With only a handful of subjects ever appearing in the corpus the steering signal here is weak; the mechanism is still the one that, at scale, makes prompt engineering work at all.
- The injection trick is meant to illustrate the *structural* problem rather than reproduce a real attack: the model has *no concept of a trusted source*. As soon as a different prefix appears, it conditions on the new prefix and continues. At production scale, a model that retrieves text from the web or accepts user-provided document fragments would be steered by *anything* in those texts. This is not a bug — it is what conditional generation does by construction.

## 🪞 Predicting in Latent Space (JEPA)

Reconstructing tokens or pixels wastes capacity on noise the downstream task does not care about. **Joint-Embedding Predictive Architectures (JEPA)** move the prediction target into feature space instead. A teacher network encodes the unmasked input, a student network sees the masked input and predicts the teacher's representations of the masked region in feature space. The loss is the mean squared error (MSE) between predicted and target features. There is no decoder.

Two structural pieces hold the method up:

- **Teacher and student share the same encoder architecture.** The teacher's weights lag the student's by an exponential moving average (EMA). The teacher is what the student is becoming, slowly.
- **The target is in feature space**, not pixel/token space. The teacher's hidden state at a masked position is what the student tries to reproduce.

> Why does the EMA matter? What goes wrong without it?

<details><summary>Thought</summary>

Without the EMA the teacher equals the student exactly at every step. The student can then minimise the loss trivially by mapping every input to the same vector — student and teacher agree, the squared error is zero, the gradient vanishes. This is *representation collapse*. The EMA delay forces the teacher to be a slowly-moving target the student has to chase, and the student cannot collapse without dragging the teacher along, which the EMA prevents from happening instantly. Why this works is one of the open theoretical questions in the area; we will see it empirically below.
</details>


### `ema_update`

The exponential moving average update is one line of arithmetic per key. Implement it.

> What does "exponential" actually mean here? With `m = 0.99`, the teacher *should* be a smoothed snapshot of which students?

<details><summary>Thought</summary>

Unroll the update: `teacher_t = (1-m)·s_t + (1-m)·m·s_{t-1} + (1-m)·m²·s_{t-2} + ...`. Each older student weight contributes `m^k` less than the most recent — exponential decay into the past. The effective averaging window is roughly `1/(1-m)`; with `m = 0.99` that is about 100 steps, so the teacher behaves like a smoothed average of the last hundred student snapshots. The same recipe shows up well beyond JEPA: target networks in deep RL (DQN), Polyak-averaged inference weights, "model EMA" as a generic regulariser.
</details>

In [ ]:
def ema_update(teacher_params, student_params, m=0.99):
    """Per-key EMA: teacher = m * teacher + (1 - m) * student."""
    # For each key in teacher_params:
    #   new_teacher[k] = m * teacher[k] + (1 - m) * student[k]
    # Return the resulting dict.
    # YOUR CODE HERE
    pass


check_ema_update(ema_update)


### `jepa_loss`

The loss is mean squared error (MSE) between student and teacher hidden states at masked positions only. Unmasked positions do not contribute.


In [ ]:
def jepa_loss(student_h, teacher_h, mask_pos):
    """MSE between student and teacher hidden states at masked positions only."""
    # diff = (student_h - teacher_h)[mask_pos]      # (n_masked, D)
    # return float((diff ** 2).mean())
    # YOUR CODE HERE
    pass


check_jepa_loss(jepa_loss)


### Train: with EMA vs without

We run the same training loop twice. The first run uses an EMA decay of `m = 0.99` — the canonical recipe. The second run uses `m = 0`, which makes the teacher copy the student instantly and removes the delay that prevents collapse. We track the embedding variance across the batch at each step; that is the canonical collapse diagnostic.


In [ ]:
def _jepa_mask_strategy(batch, rng):
    return mask_tokens(batch, MASK_ID, vocab_size, frac=0.15, rng=rng)

if (ema_update({'a': np.ones((2,2), dtype=np.float32)},
               {'a': np.zeros((2,2), dtype=np.float32)}, m=0.5) is not None
    and jepa_loss(np.zeros((1, 2, 4), dtype=np.float32),
                  np.zeros((1, 2, 4), dtype=np.float32),
                  np.array([[True, False]])) is not None):
    s_init = init_params(vocab_size, seed=10)
    t_init = {k: v.copy() for k, v in s_init.items()}
    _, _, var_with_ema = train_jepa_run(
        s_init, t_init, ids, attention, _jepa_mask_strategy,
        ema_update, m=0.99, n_steps=80, batch_size=32, seed=10,
    )

    s_init = init_params(vocab_size, seed=10)
    t_init = {k: v.copy() for k, v in s_init.items()}
    _, _, var_no_ema = train_jepa_run(
        s_init, t_init, ids, attention, _jepa_mask_strategy,
        ema_update, m=0.0, n_steps=80, batch_size=32, seed=10,
    )

    plot_jepa_variance(var_with_ema, var_no_ema)
    print(f'  with EMA   m=0.99: variance tail = {var_with_ema[-1]:.4f}')
    print(f'  without    m=0.00: variance tail = {var_no_ema[-1]:.4f}')
    print(f'  collapse ratio (no/with) at tail: {var_no_ema[-1] / max(var_with_ema[-1], 1e-9):.3f}')
else:
    print('⬜ Need ema_update and jepa_loss implemented first.')


**Observe:**

- With the EMA on (`m = 0.99`), the student's per-batch embedding variance stays positive — different inputs produce different representations, and the student is learning a meaningful encoder.
- Without the EMA (`m = 0`), the variance collapses toward zero within a few dozen steps. The student has found the trivial solution: map every input to the same vector. Loss is zero, gradient is zero, the encoder is useless.
- The same pattern shows up across modalities and method names: I-JEPA (JEPA on images), V-JEPA (JEPA on video), and the close cousins BYOL, SimSiam, and DINO — all self-distillation methods that prevent collapse with the same trick of an asymmetric, slowly-updated target network. Modality and acronym change; the recipe does not.


### 🏁 Recap

**What we did:**

- 🏷️ Looked at the asymmetry between hand-labelled datasets and raw corpora, and saw how a single sentence becomes its own supervision under three different masking strategies.
- 🎭 Implemented scaled dot-product attention, BERT-style 80/10/10 masking, and the masked-token cross-entropy. Trained a tiny encoder language model in the browser via Adam and watched it fill in held-out windows with the right words.
- 🧩 Generalised single-token masking to *span* masking (geometric-length contiguous chunks). Same architecture, harder pretext, longer-range context required to solve it.
- 📜 Switched the attention pattern to causal, the loss to next-token cross-entropy, and the inference path to autoregressive sampling. Same architecture, decoder-only model. Used prefix steering and a mid-stream injection demo to expose the structural reason prompt injection exists.
- 🪞 Predicted in feature space instead of token space. Two runs of the same training loop, with and without the exponential moving average teacher update, made representation collapse appear as a clean line on a single plot.

**Key takeaways:**

- Self-supervision turns the *structure of the data* into the supervision signal. We never wrote down a label; the masking strategy was the label.
- Encoder-only (no attention mask), decoder-only (causal mask), and encoder-decoder are three architectural recipes built on the same denoising objective. The attention-mask pattern decides which one you are training.
- Span masking forces longer-range integration than per-token masking at the same mask fraction. The harder pretext usually transfers better.
- Generation is conditional sampling. Conditional sampling has no idea what a "trusted prompt" is, which is why prompt injection is *not* a bug — it is the system working as designed.
- Predicting in latent space avoids the cost of reconstructing pixels or tokens, but is only stable when the teacher lags the student. The EMA delay is what stops the encoder from collapsing onto a constant.

The next chapter, semi-supervised and self-training methods, closes the loop with another way of squeezing supervision out of unlabelled data: letting the model label *itself*.

## 🧵 Take It from Here: stitch a real LLM into a real coding tool

Last lecture you got [Ollama](https://ollama.com) running locally and pulled a small open-weights model — your own foundation model, no API key, no cloud round-trip. This time we hand that local model to a *real* coding interface and feel what the recipe from this chapter (self-supervised pretraining + masked / next-token prediction) does once it is wired into an actual tool you might use day to day.

The bridge is **[Aider](https://aider.chat)** — an open-source AI pair programmer that runs in your terminal, edits files in place, and commits every change to git. Install it, point it at the Ollama daemon you already have, and use it on whatever project of yours feels relevant — a side project, a coursework repo, this notebook, anything. No new model gets trained.

**Setup**

1. **Install Aider** (recommended via `pipx` so it lives in its own environment):
   ```
   pipx install aider-chat
   ```
   Or via pip:
   ```
   pip install aider-chat
   ```

2. **Make sure Ollama is running** and pull a small code-focused model:
   ```
   ollama pull qwen2.5-coder:0.5b    # ~400 MB, the smallest worth trying
   # or, a step up:
   ollama pull qwen2.5-coder:1.5b    # ~1 GB, noticeably better
   ```
   Bigger Qwen-Coder variants exist (3B, 7B, 14B, 32B) and the answers improve sharply with each step — but we want to start where the model fits comfortably on a laptop.

3. **Launch Aider** inside any git repository you are working in, pointing it at the local Ollama model and whichever file(s) you want it to be able to read and edit:
   ```
   cd ~/path/to/your/project
   aider --model ollama/qwen2.5-coder:0.5b path/to/file.py
   ```
   That is it — no JSON config, no provider boilerplate. The `ollama/` prefix tells Aider to talk to the local daemon at `localhost:11434`. If you are outside a git repo, run `git init` first; Aider commits every edit it makes, which is half the demo.

**Exercise: drive Aider on a real codebase of yours**

Pick something — a side project, a coursework repo, this lecture's notebook, whatever. A few categories of prompt that exercise different aspects of the loop:

- *Explanation*: *"Walk me through what `<some_function>` is doing and why it is structured this way."*
- *Documentation*: *"Add a docstring to `<some_function>` that captures the parameters and the failure modes."*
- *Comparison*: *"We use approach A in module X and approach B in module Y for the same kind of thing. Why? What would change if I unified them?"*
- *Free prose*: *"Draft a one-paragraph README note explaining why this project exists."*

After each turn, run `git log --oneline` and `git diff HEAD~1` to see exactly what the model changed. The commit-per-turn workflow is half the pedagogical point — the model's edits stop being a black box.

> Why does the same model feel sharper inside Aider than when you poked it from a one-shot `requests.post` call last lecture?

<details><summary>Thought</summary>

Aider is doing three things behind the scenes that a one-shot HTTP call is not. *Context*: it feeds the model the file contents, the repo structure, and prior conversation as part of the prompt — the model is conditioning on a much richer prefix. *System prompt*: Aider injects an instruction prompt that tells the model to return diffs in a specific format, ask before editing, and stay focused — the same prefix-steering mechanic from section 📜, used productively. *Loop*: each turn Aider applies the model's diff, commits it, and lets the model see the result on the next turn, so individual responses only have to handle one step. None of this changes the underlying network. It changes the *prompt* the model sees, which is the only steering surface a self-supervised LLM has.
</details>

**What you should notice**

- A 0.5B-parameter model running on your CPU, talking to your local files via Aider, *should* already deliver something genuinely useful. The only ingredient between this and a professional setup is *scale* — the recipe (self-supervised pretraining + a UI on top) is identical.
- The model will sometimes hallucinate or get the diff format wrong. It does not understand the project; it is conditioning on the prompt and the files it sees. The same prompt-injection / prompt-steering mechanic from section 📜, this time without the small-corpus caveat — every word in the prefix steers the next token.
- The latency, the failure modes, and the quality of answers will shift a lot when you swap in a bigger model (`aider --model ollama/qwen2.5-coder:1.5b`, then `3b`, then `7b` if your machine has the RAM). The architecture stays identical; only the parameter count moves. That is exactly the scaling behaviour that makes self-supervised pretraining the dominant recipe.